# Trust Drift PoC

In [80]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [81]:
RAW_PATH     = 'trust_drift/data/raw'
CLEANED_PATH = 'trust_drift/data/cleaned'

FILES = {
    'monday'    : 'Monday-WorkingHours.pcap_ISCX.csv',
    'tuesday'   : 'Tuesday-WorkingHours.pcap_ISCX.csv',
    'wednesday' : 'Wednesday-workingHours.pcap_ISCX.csv'
}


In [82]:
monday_data    = pd.read_csv(os.path.join(RAW_PATH, FILES['monday']))
tuesday_data   = pd.read_csv(os.path.join(RAW_PATH, FILES['tuesday']))
wednesday_data = pd.read_csv(os.path.join(RAW_PATH, FILES['wednesday']))

print(f'\nMonday    -  {len(monday_data):,} rows,  {len(monday_data.columns)} columns')
print(f'Tuesday   -  {len(tuesday_data):,} rows,  {len(tuesday_data.columns)} columns')
print(f'Wednesday -  {len(wednesday_data):,} rows,  {len(wednesday_data.columns)} columns')
print('\nRaw data loaded!')


Monday    -  529,918 rows,  79 columns
Tuesday   -  445,909 rows,  79 columns
Wednesday -  692,703 rows,  79 columns

Raw data loaded!


In [83]:
def analyse_data(df, name):
    df.columns = df.columns.str.strip()
    
    print(f'\n{"="*50}')
    print(f'Dataset: {name}')
    print(f'{"="*50}')
    
    # Basic info
    print(f'Rows:     {len(df):,}')
    print(f'Columns:  {len(df.columns)}')
    
    # Check null values
    null_counts = df.isnull().sum()
    null_cols   = null_counts[null_counts > 0]
    print(f'\nNull Values:')
    if len(null_cols) == 0:
        print('None found')
    else:
        print(f'  {null_cols.to_dict()}')
        print(f'  Total: {null_counts.sum()}')
    
    # Check infinity values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_counts   = np.isinf(df[numeric_cols]).sum()
    inf_cols     = inf_counts[inf_counts > 0]
    print(f'\nInfinity Values:')
    if len(inf_cols) == 0:
        print('None found')
    else:
        print(f'  {inf_cols.to_dict()}')
        print(f'  Total: {inf_counts.sum()}')
    
    # Check labels
    print(f'\nLabels:')
    for label, count in df['Label'].value_counts().items():
        print(f'  {label:<30} →  {count:,}')
    
    return df

monday_data    = analyse_data(monday_data,    'Monday')
tuesday_data   = analyse_data(tuesday_data,   'Tuesday')
wednesday_data = analyse_data(wednesday_data, 'Wednesday')


Dataset: Monday
Rows:     529,918
Columns:  79

Null Values:
  {'Flow Bytes/s': 64}
  Total: 64

Infinity Values:
  {'Flow Bytes/s': 373, 'Flow Packets/s': 437}
  Total: 810

Labels:
  BENIGN                         →  529,918

Dataset: Tuesday
Rows:     445,909
Columns:  79

Null Values:
  {'Flow Bytes/s': 201}
  Total: 201

Infinity Values:
  {'Flow Bytes/s': 63, 'Flow Packets/s': 264}
  Total: 327

Labels:
  BENIGN                         →  432,074
  FTP-Patator                    →  7,938
  SSH-Patator                    →  5,897

Dataset: Wednesday
Rows:     692,703
Columns:  79

Null Values:
  {'Flow Bytes/s': 1008}
  Total: 1008

Infinity Values:
  {'Flow Bytes/s': 289, 'Flow Packets/s': 1297}
  Total: 1586

Labels:
  BENIGN                         →  440,031
  DoS Hulk                       →  231,073
  DoS GoldenEye                  →  10,293
  DoS slowloris                  →  5,796
  DoS Slowhttptest               →  5,499
  Heartbleed                     →  11


In [ ]:
def clean_data(df, name):
    print(f'\nCleaning {name}...')

    df.columns = df.columns.str.strip()
    
    df = df.replace([np.inf, -np.inf], np.nan)

    before = len(df)
    df     = df.dropna()
    after  = len(df)
    
    print(f'  Rows before:  {before:,}')
    print(f'  Rows after:   {after:,}')
    print(f'  Rows removed: {before - after:,}  ({(before-after)/before*100:.3f}%)')
    print(f'  Labels:       {df["Label"].value_counts().to_dict()}')
    print(f'  Status:       Clean')
    
    return df

# ── Clean all 3 datasets
monday_clean    = clean_data(monday_data,    'Monday')
tuesday_clean   = clean_data(tuesday_data,   'Tuesday')
wednesday_clean = clean_data(wednesday_data, 'Wednesday')


Cleaning Monday...
  Rows before:  529,918
  Rows after:   529,481
  Rows removed: 437  (0.082%)
  Labels:       {'BENIGN': 529481}
  Status:       Clean

Cleaning Tuesday...
  Rows before:  445,909
  Rows after:   445,645
  Rows removed: 264  (0.059%)
  Labels:       {'BENIGN': 431813, 'FTP-Patator': 7935, 'SSH-Patator': 5897}
  Status:       Clean

Cleaning Wednesday...
  Rows before:  692,703
  Rows after:   691,406
  Rows removed: 1,297  (0.187%)
  Labels:       {'BENIGN': 439683, 'DoS Hulk': 230124, 'DoS GoldenEye': 10293, 'DoS slowloris': 5796, 'DoS Slowhttptest': 5499, 'Heartbleed': 11}
  Status:       Clean


In [85]:
monday_clean.to_csv(os.path.join(CLEANED_PATH, 'monday_clean.csv'),       index=False)
tuesday_clean.to_csv(os.path.join(CLEANED_PATH, 'tuesday_clean.csv'),     index=False)
wednesday_clean.to_csv(os.path.join(CLEANED_PATH, 'wednesday_clean.csv'), index=False)

print('\nSaved files:')
for f in os.listdir(CLEANED_PATH):
    size = os.path.getsize(os.path.join(CLEANED_PATH, f)) / (1024*1024)
    print(f'  {f}  -  {size:.1f} MB')


Saved files:
  .gitkeep  -  0.0 MB
  monday_clean.csv  -  182.9 MB
  tuesday_clean.csv  -  141.1 MB
  wednesday_clean.csv  -  232.8 MB
